In [1]:
import sys
!{sys.executable} -m pip install pandas


In [2]:
import sys
print(sys.executable)
print(sys.version)


c:\Users\moham\AppData\Local\Programs\Python\Python312\python.exe
3.12.6 (tags/v3.12.6:a4a2d2b, Sep  6 2024, 20:11:23) [MSC v.1940 64 bit (AMD64)]


In [3]:
import pandas as pd
pd.__version__


'2.3.3'

In [5]:
!pip install pandas

In [6]:
import pandas as pd

# Load your dataset
df = pd.read_csv("PROMISE-relabeled-NICE.csv")

# Show before changes
print("Before adding Type column:")
print(df.head())

# Define the NFR columns (the binary 0/1 ones)
nfr_columns = [
    "IsFunctional","Availability (A)", "Fault Tolerance (FT)", "Legal (L)", "Look & Feel (LF)",
    "Maintainability (MN)", "Operability (O)", "Performance (PE)", "Portability (PO)",
    "Scalability (SC)", "Security (SE)", "Usability (US)", "Other (OT)"
]

# Create the new 'Type' column
def get_type(row):
    types = [col for col in nfr_columns if row[col] == 1]
    # Remove the short code in parentheses for cleaner output
    return ", ".join([col.split(" (")[0] for col in types]) if types else "None"

df["Type"] = df.apply(get_type, axis=1)

# Drop unnecessary columns
columns_to_drop = ["ProjectID","IsQuality"] + nfr_columns
df.drop(columns=columns_to_drop, inplace=True, errors="ignore")

# Save to a new file
df.to_csv("merged_dataset_cleaned.csv", index=False)

# Show the result
print("\nAfter cleaning and adding Type column:")
print(df.head())


Before adding Type column:
   ProjectID                                    RequirementText  IsFunctional  \
0          1  'The system shall refresh the display every 60...             1   
1          1  'The application shall match the color of the ...             0   
2          1  'If projected the data must be readable. On a ...             0   
3          1  'The product shall be available during normal ...             0   
4          1  'If projected the data must be understandable....             0   

   IsQuality  Availability (A)  Fault Tolerance (FT)  Legal (L)  \
0          1                 0                     0          0   
1          1                 0                     0          1   
2          1                 0                     0          0   
3          1                 1                     0          0   
4          1                 0                     0          0   

   Look & Feel (LF)  Maintainability (MN)  Operability (O)  Performance (PE)  \
0  

In [7]:
import pandas as pd

# Load the cleaned dataset
df = pd.read_csv("merged_dataset_cleaned.csv")

# Rename 'RequirementText' to 'Requirement'
df.rename(columns={"RequirementText": "Requirement"}, inplace=True)

# Split rows where 'Type' has multiple values (e.g., "Performance, Security")
df["Type"] = df["Type"].fillna("None")  # Handle any missing values
df = df.assign(Type=df["Type"].str.split(", ")).explode("Type").reset_index(drop=True)

# Save the final dataset
df.to_csv("merged_dataset_final.csv", index=False)

# Show a preview
print("✅ Cleaned and normalized dataset:")
print(df.head(10))


✅ Cleaned and normalized dataset:
                                         Requirement          Type
0  'The system shall refresh the display every 60...  IsFunctional
1  'The system shall refresh the display every 60...   Performance
2  'The application shall match the color of the ...         Legal
3  'The application shall match the color of the ...   Look & Feel
4  'If projected the data must be readable. On a ...   Look & Feel
5  'If projected the data must be readable. On a ...     Usability
6  'The product shall be available during normal ...  Availability
7  'If projected the data must be understandable....   Look & Feel
8  'If projected the data must be understandable....     Usability
9  'The product shall ensure that it can only be ...  IsFunctional


In [8]:
import pandas as pd

# Load the final dataset
df = pd.read_csv("merged_dataset_final.csv")

# Define the mapping dictionary
type_mapping = {
    "IsFunctional": "F",
    "Functional Requirement": "FR",
    "Availability": "A",
    "Fault Tolerance": "FT",
    "Legal": "L",
    "Look & Feel": "LF",
    "Maintainability": "MN",
    "Operational": "O",
    "Operability": "O",  # In case some use "Operability"
    "Performance": "PE",
    "Portability": "PO",
    "Scalability": "SC",
    "Security": "SE",
    "Usability": "US",
    "Non-Functional Requirement": "NFR"
}

# Apply the mapping to the 'Type' column
df["Type"] = df["Type"].map(type_mapping).fillna(df["Type"])

# Save the renamed dataset
df.to_csv("merged_dataset_final_abbreviated.csv", index=False)

# Preview result
print("✅ Type values abbreviated successfully:")
print(df["Type"].unique())


✅ Type values abbreviated successfully:
['F' 'PE' 'L' 'LF' 'US' 'A' 'SE' nan 'MN' 'PO' 'FT' 'SC' 'O']


In [9]:
import pandas as pd

# Load the 4th dataset
df4 = pd.read_csv("nfr_training_data.csv")  # Replace with actual filename

# Rename columns
df4 = df4.rename(columns={"category": "Type", "text": "Requirement"})

# Drop the 'label' column if it exists
if "label" in df4.columns:
    df4 = df4.drop(columns=["label"])

# Map full names to abbreviations
type_mapping = {
    "Functional": "F",
    "Functional Requirement": "FR",
    "Availability": "A",
    "Fault Tolerance": "FT",
    "Legal": "L",
    "Look & Feel": "LF",
    "Maintainability": "MN",
    "Operational": "O",
    "Performance": "PE",
    "Portability": "PO",
    "Scalability": "SC",
    "Security": "SE",
    "Usability": "US",
    "Non-Functional Requirement": "NFR"
}

# Replace Type column content
df4["Type"] = df4["Type"].map(type_mapping)

# Keep only Type and Requirement
df4 = df4[["Type", "Requirement"]]

# Save to Dataset3.csv
df4.to_csv("Dataset3.csv", index=False)

print("✅ Dataset3.csv prepared successfully with renamed Type column")
print(df4.head())


✅ Dataset3.csv prepared successfully with renamed Type column
  Type                                        Requirement
0   PE  The system must provide a fast and efficient p...
1   PE  CV parsing speed must be optimized to under 2 ...
2   PE       System should respond in less than 1 second.
3   PE            System performance can vary under load.
4   PE       Performance is not critical for this system.


In [10]:
import pandas as pd

# Load your dataset
df = pd.read_csv("PROMISE-relabeled-NICE.csv")

# Show first few rows
print("Before adding Type column:")
print(df.head())

# List all NFR columns (the ones with 0/1 values)
nfr_columns = [
    "Availability (A)", "Fault Tolerance (FT)", "Legal (L)", "Look & Feel (LF)",
    "Maintainability (MN)", "Operability (O)", "Performance (PE)", "Portability (PO)",
    "Scalability (SC)", "Security (SE)", "Usability (US)", "Other (OT)"
]

# Create the new 'Type' column
def get_type(row):
    # Get the column(s) where value == 1
    types = [col for col in nfr_columns if row[col] == 1]
    # Join multiple ones (if a row has more than one NFR type)
    return ", ".join(types) if types else "None"

df["Type"] = df.apply(get_type, axis=1)

# Save to a new file (optional)
df.to_csv("merged_dataset_with_type.csv", index=False)

# Show the result
print("\nAfter adding Type column:")
print(df[["RequirementText", "Type"]].head())


Before adding Type column:
   ProjectID                                    RequirementText  IsFunctional  \
0          1  'The system shall refresh the display every 60...             1   
1          1  'The application shall match the color of the ...             0   
2          1  'If projected the data must be readable. On a ...             0   
3          1  'The product shall be available during normal ...             0   
4          1  'If projected the data must be understandable....             0   

   IsQuality  Availability (A)  Fault Tolerance (FT)  Legal (L)  \
0          1                 0                     0          0   
1          1                 0                     0          1   
2          1                 0                     0          0   
3          1                 1                     0          0   
4          1                 0                     0          0   

   Look & Feel (LF)  Maintainability (MN)  Operability (O)  Performance (PE)  \
0  

In [11]:
# Load the other three datasets
df1 = pd.read_csv("nfr_dataset.csv")[["Type", "Requirement"]]
df2 = pd.read_csv("Dataset2.csv")[["Type", "Requirement"]]
df3 = pd.read_csv("merged_dataset_final_abbreviated.csv")[["Type", "Requirement"]]
df4 = pd.read_csv("Dataset3.csv")[["Type", "Requirement"]]

# Concatenate all datasets
merged_df = pd.concat([df1, df2, df3, df4], ignore_index=True)

# Optional: remove duplicates
merged_df = merged_df.drop_duplicates().reset_index(drop=True)

# Save final merged dataset
merged_df.to_csv("merged_NFR.csv", index=False)

print("✅ All datasets merged into merged_NFR.csv")
print(merged_df.head(10))
print(f"Total rows: {len(merged_df)}")


✅ All datasets merged into merged_NFR.csv
  Type                                        Requirement
0   PE  The system shall refresh the display every 60 ...
1   LF  The application shall match the color of the s...
2   US   If projected  the data must be readable.  On ...
3    A   The product shall be available during normal ...
4   US   If projected  the data must be understandable...
5   SE  The product shall ensure that it can only be a...
6   US  The product shall be intuitive and self-explan...
7   PE  The product shall respond fast to keep up-to-d...
8    F  The system shall have a MDI form that allows f...
9    F  The system shall display Events in a vertical ...
Total rows: 2466


In [12]:
import pandas as pd

# Load the merged file
df = pd.read_csv("merged_NFR.csv")

# 1. Remove rows where Type is empty or NaN
df = df[df['Type'].notna() & (df['Type'] != '')]

# 2. Remove rows where Type is F, FR, or NFR
df = df[~df['Type'].isin(['F', 'FR', 'NFR'])]

# 3. Remove duplicated rows (consider both Type and Requirement)
df = df.drop_duplicates(subset=['Type', 'Requirement'])

# Save the cleaned file
df.to_csv("merged_NFR_cleaned.csv", index=False)

print("✅ merged_NFR.csv cleaned successfully")
print(df.head())


✅ merged_NFR.csv cleaned successfully
  Type                                        Requirement
0   PE  The system shall refresh the display every 60 ...
1   LF  The application shall match the color of the s...
2   US   If projected  the data must be readable.  On ...
3    A   The product shall be available during normal ...
4   US   If projected  the data must be understandable...


In [13]:
import pandas as pd

# Load the cleaned dataset
df = pd.read_csv("merged_NFR_cleaned.csv")

# Add an empty 'Level' column
df["Level"] = ""

# Save the updated file
df.to_csv("merged_NFR_cleaned1.csv", index=False)

print("✅ 'Level' column added (empty for now).")
print(df.head())


✅ 'Level' column added (empty for now).
  Type                                        Requirement Level
0   PE  The system shall refresh the display every 60 ...      
1   LF  The application shall match the color of the s...      
2   US   If projected  the data must be readable.  On ...      
3    A   The product shall be available during normal ...      
4   US   If projected  the data must be understandable...      


In [14]:
import pandas as pd

# Load your CSV
df = pd.read_csv("merged_NFR_cleaned1.csv", quotechar='"', on_bad_lines='skip')

# Remove dots from all string/object columns
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.replace('.', '', regex=False)

# Optional: save the cleaned dataset
df.to_csv("merged_NFR_cleaned_no_dots.csv", index=False)

print("Dots removed from dataset successfully!")


Dots removed from dataset successfully!


In [15]:
!pip install --upgrade accelerate transformers[torch]

In [16]:
!pip install datasets
!pip install evaluate
!pip install --upgrade transformers




#hena el ordinal

In [17]:
import sys
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install torch


In [18]:
import sys
print("Kernel Python:", sys.executable)

!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install transformers datasets evaluate accelerate


Kernel Python: c:\Users\moham\AppData\Local\Programs\Python\Python312\python.exe


In [19]:
import sys
!{sys.executable} -m pip install scikit-learn


In [ ]:
# ===========================
# NFR Extraction using BERT from merged_NFR_cleaned.csv
# ===========================

# Install required packages if not installed:
# pip install torch transformers datasets scikit-learn evaluate pandas

import pandas as pd
import numpy as np
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from datasets import Dataset
import evaluate
from sklearn.preprocessing import LabelEncoder

# ---------------------------
# 1. Load CSV
# ---------------------------
df = pd.read_csv("merged_NFR_cleaned.csv")

print("Sample of loaded dataset:")
print(df.head())

# Encode labels
le = LabelEncoder()
df['label_enc'] = le.fit_transform(df['Type'])  # numeric labels

# ---------------------------
# 2. Convert to HuggingFace Dataset
# ---------------------------
dataset = Dataset.from_pandas(df[['Requirement', 'label_enc']])
dataset = dataset.rename_column("label_enc", "label")

# Split into train/test (80/20)
dataset = dataset.train_test_split(test_size=0.2)
train_dataset = dataset['train']
test_dataset = dataset['test']

# ---------------------------
# 3. Load BERT tokenizer
# ---------------------------
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(batch['Requirement'], padding=True, truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

# Set dataset format for PyTorch
train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

# ---------------------------
# 4. Load BERT model
# ---------------------------
num_labels = len(le.classes_)
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=num_labels)

# ---------------------------
# 5. Metrics
# ---------------------------
metric_accuracy = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = metric_accuracy.compute(predictions=predictions, references=labels)['accuracy']
    f1 = metric_f1.compute(predictions=predictions, references=labels, average='weighted')['f1']
    return {"accuracy": acc, "f1": f1}

# ---------------------------
# 6. Data Collator for dynamic padding
# ---------------------------
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# ---------------------------
# 7. Training arguments
# ---------------------------
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./logs",
    logging_steps=10,
    logging_first_step=True
    # Note: no evaluation_strategy or save_strategy in old versions
)


# ---------------------------
# 8. Trainer
# ---------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
    data_collator=data_collator
)

# ---------------------------
# 9. Train model
# ---------------------------
trainer.train()
# Save the trained model and tokenizer
model.save_pretrained(
    "./trained_nfr_model",
    safe_serialization=False
)

tokenizer.save_pretrained("./trained_nfr_model")

# ---------------------------
# 10. Evaluate model
# ---------------------------
eval_results = trainer.evaluate()
print("Evaluation results:", eval_results)

# ---------------------------
# 11. Prediction example
# ---------------------------
texts = [
    "A web-based e-commerce platform initially supports 1,000 users per hour.",
    "A search engine returns results for a query in under 2 seconds"
]

tokens = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
with torch.no_grad():
    outputs = model(**tokens)
    preds = torch.argmax(outputs.logits, dim=1)

pred_labels = le.inverse_transform(preds.numpy())
print("Predicted NFR categories:", pred_labels.tolist())

c:\Users\moham\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Sample of loaded dataset:
  Type                                        Requirement
0   PE  The system shall refresh the display every 60 ...
1   LF  The application shall match the color of the s...
2   US   If projected  the data must be readable.  On ...
3    A   The product shall be available during normal ...
4   US   If projected  the data must be understandable...


Map: 100%|██████████| 293/293 [00:00<00:00, 848.12 examples/s]
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\moham\AppData\Local\Temp\ipykernel_15412\160441726.py:96: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
c:\Users\moham\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
1,2.496400
10,2.414200
20,2.310100
30,2.256500
40,2.101300
50,1.978500
60,1.945500
70,1.897700
80,1.700500
90,1.467200


In [27]:
# ===========================
# NFR Extraction using BERT for both Type and Level
# ===========================

# !pip install torch transformers datasets scikit-learn evaluate pandas

import pandas as pd
import numpy as np
import torch
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import Dataset
import evaluate
from sklearn.preprocessing import LabelEncoder

# ---------------------------
# 1. Load CSV
# ---------------------------
df = pd.read_csv("merged_NFR_cleaned_no_dots.csv")

print("Sample of loaded dataset:")
print(df.head())

# ---------------------------
# 2. Encode both Type and Level
# ---------------------------
le_type = LabelEncoder()
le_level = LabelEncoder()

df['type_enc'] = le_type.fit_transform(df['Type'])
df['level_enc'] = le_level.fit_transform(df['Level'])

# ---------------------------
# 3. Initialize tokenizer
# ---------------------------
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(batch['Requirement'], padding=True, truncation=True, max_length=128)

# ---------------------------
# 4. Metrics
# ---------------------------
metric_accuracy = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = metric_accuracy.compute(predictions=predictions, references=labels)['accuracy']
    f1 = metric_f1.compute(predictions=predictions, references=labels, average='weighted')['f1']
    return {"accuracy": acc, "f1": f1}

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# ---------------------------
# 5. Training arguments (shared)
# ---------------------------
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./logs",
    logging_steps=10,
    logging_first_step=True
)

# ===================================================
# ========== MODEL 1: Train for TYPE ================
# ===================================================
print("\n========== Training model for TYPE ==========")

dataset_type = Dataset.from_pandas(df[['Requirement', 'type_enc']])
dataset_type = dataset_type.rename_column("type_enc", "label")
dataset_type = dataset_type.train_test_split(test_size=0.2)

train_type = dataset_type['train'].map(tokenize, batched=True)
test_type = dataset_type['test'].map(tokenize, batched=True)

train_type.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_type.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

num_labels_type = len(le_type.classes_)
model_type = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=num_labels_type)

trainer_type = Trainer(
    model=model_type,
    args=training_args,
    train_dataset=train_type,
    eval_dataset=test_type,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
    data_collator=data_collator
)

trainer_type.train()
model_type.save_pretrained(
    "./trained_nfr_type_model",
    safe_serialization=False
)

tokenizer.save_pretrained("./trained_nfr_type_model")

eval_results_type = trainer_type.evaluate()
print("TYPE model evaluation:", eval_results_type)


# ===================================================
# ========== MODEL 2: Train for LEVEL ===============
# ===================================================
print("\n========== Training model for LEVEL ==========")

dataset_level = Dataset.from_pandas(df[['Requirement', 'level_enc']])
dataset_level = dataset_level.rename_column("level_enc", "label")
dataset_level = dataset_level.train_test_split(test_size=0.2)

train_level = dataset_level['train'].map(tokenize, batched=True)
test_level = dataset_level['test'].map(tokenize, batched=True)

train_level.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_level.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

num_labels_level = len(le_level.classes_)
model_level = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=num_labels_level)

trainer_level = Trainer(
    model=model_level,
    args=training_args,
    train_dataset=train_level,
    eval_dataset=test_level,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
    data_collator=data_collator
)

trainer_level.train()
model_level.save_pretrained("./trained_nfr_level_model")
tokenizer.save_pretrained("./trained_nfr_level_model")

eval_results_level = trainer_level.evaluate()
print("LEVEL model evaluation:", eval_results_level)


# ===================================================
# ========== 11. Combined Prediction Example =========
# ===================================================
texts = [
    "A web-based e-commerce platform initially supports 1,000 users per hour.",
    "A search engine returns results for a query in under 2 seconds"
]

tokens = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")

with torch.no_grad():
    outputs_type = model_type(**tokens)
    preds_type = torch.argmax(outputs_type.logits, dim=1)

    outputs_level = model_level(**tokens)
    preds_level = torch.argmax(outputs_level.logits, dim=1)

pred_labels_type = le_type.inverse_transform(preds_type.numpy())
pred_labels_level = le_level.inverse_transform(preds_level.numpy())

print("\n========== PREDICTION EXAMPLES ==========")
for req, t, l in zip(texts, pred_labels_type, pred_labels_level):
    print(f"Requirement: {req}")
    print(f" → Type: {t}, Level: {l}\n")


Sample of loaded dataset:
  Type                                        Requirement   Level
0   PE  The system shall refresh the display every 60 ...    High
1   LF  The application shall match the color of the s...     Low
2   US  If projected  the data must be readable  On a ...  Medium
3    A  The product shall be available during normal b...    High
4   US  If projected  the data must be understandable ...  Medium

========== Training model for TYPE ==========


Map: 100%|██████████| 288/288 [00:00<00:00, 3221.63 examples/s]
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\moham\AppData\Local\Temp\ipykernel_8824\393158975.py:92: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_type = Trainer(
c:\Users\moham\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
1,2.606000
10,2.407000
20,2.371300
30,2.362200
40,2.178200
50,2.077400
60,2.193700
70,2.042100
80,1.832900
90,1.586300


c:\Users\moham\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TYPE model evaluation: {'eval_loss': 0.8503656387329102, 'eval_accuracy': 0.7638888888888888, 'eval_f1': 0.7498876161087371, 'eval_runtime': 29.2855, 'eval_samples_per_second': 9.834, 'eval_steps_per_second': 1.229, 'epoch': 3.0}

========== Training model for LEVEL ==========


Map: 100%|██████████| 288/288 [00:00<00:00, 838.70 examples/s]
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\moham\AppData\Local\Temp\ipykernel_8824\393158975.py:132: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_level = Trainer(
c:\Users\moham\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
1,1.797900
10,1.752500
20,1.719000
30,1.733800
40,1.597900
50,1.574800
60,1.648400
70,1.591700
80,1.575300
90,1.392100


c:\Users\moham\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


LEVEL model evaluation: {'eval_loss': 0.9185375571250916, 'eval_accuracy': 0.6805555555555556, 'eval_f1': 0.6753333581094737, 'eval_runtime': 28.2681, 'eval_samples_per_second': 10.188, 'eval_steps_per_second': 1.274, 'epoch': 3.0}

========== PREDICTION EXAMPLES ==========
Requirement: A web-based e-commerce platform initially supports 1,000 users per hour.
 → Type: SC, Level: Medium

Requirement: A search engine returns results for a query in under 2 seconds
 → Type: PE, Level: High



In [28]:
import pandas as pd
import torch
import numpy as np
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.preprocessing import LabelEncoder
import json


def predict_nfr_types_and_levels(
    csv_path="merged_NFR_cleaned_no_dots.csv",
    json_path="non_functional_requirements.json",
    model_type_path="./trained_nfr_type_model",
    model_level_path="./trained_nfr_level_model",
    output_json="nfr_predictions_type_level.json"
):
    """
    Predict Type + Level for NFR JSON using pre-trained BERT models.
    """

    # -------------------------
    # 1. Load CSV + Fit Encoders
    # -------------------------
    df = pd.read_csv(csv_path)

    le_type = LabelEncoder()
    le_type.fit(df['Type'])

    le_level = LabelEncoder()
    le_level.fit(df['Level'])

    # -------------------------
    # 2. Load Models + Tokenizer
    # -------------------------
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

    model_type = BertForSequenceClassification.from_pretrained(model_type_path)
    model_level = BertForSequenceClassification.from_pretrained(model_level_path)

    model_type.eval()
    model_level.eval()

    # -------------------------
    # 3. Load NFR JSON
    # -------------------------
    with open(json_path, "r", encoding="utf-8") as f:
        nfrs = json.load(f)

    texts = [item["description"] for item in nfrs]

    # -------------------------
    # 4. Tokenize once
    # -------------------------
    tokens = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")

    # -------------------------
    # 5. Predict Type
    # -------------------------
    with torch.no_grad():
        outputs_type = model_type(**tokens)
        preds_type = torch.argmax(outputs_type.logits, dim=1)

    predicted_types = le_type.inverse_transform(preds_type.numpy())

    # -------------------------
    # 6. Predict Level
    # -------------------------
    with torch.no_grad():
        outputs_level = model_level(**tokens)
        preds_level = torch.argmax(outputs_level.logits, dim=1)

    predicted_levels = le_level.inverse_transform(preds_level.numpy())

    # -------------------------
    # 7. Merge Predictions
    # -------------------------
    results = []
    for i, item in enumerate(nfrs):
        results.append({
            "title": item["title"],
            "description": item["description"],
            "predicted_type": predicted_types[i],
            "predicted_level": predicted_levels[i]
        })

    # -------------------------
    # 8. Save Output JSON
    # -------------------------
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    # -------------------------
    # 9. Return Results
    # -------------------------
    return results

In [29]:
import pandas as pd
import json

def recommend_architectures(prediction_file="nfr_predictions_type_level.json",
                            dataset_file="ArchitectureDataset.csv",
                            output_file="Ordinal_Method_Top_Arch.json",
                            top_k=5):
    """
    Recommend architecture styles based on predicted NFR (Type + Level).
    
    Parameters:
        prediction_file (str): JSON file containing predicted_type + predicted_level
        dataset_file (str): CSV file mapping (Type, Level) -> Architecture
        output_file (str): JSON file to save top architectures
        top_k (int): number of top architectures to return
    
    Returns:
        list: list of dictionaries with architecture style and score
    """

    # ============================================
    # 1. Load Predictions
    # ============================================
    with open(prediction_file, "r", encoding="utf-8") as f:
        predictions = json.load(f)

    pred_df = pd.DataFrame(predictions)

    # Ensure consistent column names
    pred_df = pred_df.rename(columns={
        "predicted_type": "Type",
        "predicted_level": "Level"
    })

    # ============================================
    # 2. Load Architecture Dataset
    # ============================================
    arch_df = pd.read_csv(dataset_file)

    arch_df = arch_df.rename(columns={
        "architecture_style": "ArchitectureStyle",
        "architecture style": "ArchitectureStyle",
        "architecture": "ArchitectureStyle"
    })

    # ============================================
    # 3. Merge predictions with dataset
    # ============================================
    matches = pred_df.merge(
        arch_df,
        on=["Type", "Level"],
        how="inner"
    )

    # ============================================
    # 4. Count matches per architecture
    # ============================================
    style_scores = matches["Architecture"].value_counts()

    # ============================================
    # 5. Get Top-K Recommendations
    # ============================================
    top_arch = style_scores.head(top_k)

    # Convert to list of dicts
    result = [
        {"Architecture": style, "MatchedNFRs": int(score)}
        for style, score in top_arch.items()
    ]

    # ============================================
    # 6. Save Results
    # ============================================
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    return result


In [30]:
# Run the NFR Type + Level prediction function
results = predict_nfr_types_and_levels(
    csv_path="merged_NFR_cleaned_no_dots.csv",
    json_path="non_functional_requirements.json",
    model_type_path="./trained_nfr_type_model",
    model_level_path="./trained_nfr_level_model",
    output_json="nfr_predictions_type_level.json"
)

# Print results
print("========== PREDICTED NFR TYPES & LEVELS ==========")
for r in results:
    print(f"{r['title']}  →  {r['predicted_type']}  (Level: {r['predicted_level']})")

print("\nSaved predictions to nfr_predictions_type_level.json")

# Call the architecture recommendation function
results = recommend_architectures(
    prediction_file="nfr_predictions_type_level.json",
    dataset_file="ArchitectureDataset.csv",
    output_file="Ordinal_Method_Top_Arch.json",
    top_k=5
)

# Print results
print("========== TOP ARCHITECTURE STYLES ==========")
for item in results:
    print(f"{item['Architecture']} → {item['MatchedNFRs']} matched NFRs")

print("\nSaved results to Ordinal_Method_Top_Arch.json")


========== PREDICTED NFR TYPES & LEVELS ==========
Standard Compliance  →  SE  (Level: High)
Accessibility Constraints  →  US  (Level: Medium)
Device Compatibility  →  PO  (Level: Medium)
Data Privacy Compliance  →  SE  (Level: High)
Scalability  →  SC  (Level: Medium)
Usability  →  US  (Level: Medium)
Performance  →  PE  (Level: High)
Security  →  SE  (Level: High)
Reliability  →  A  (Level: High)

Saved predictions to nfr_predictions_type_level.json
========== TOP ARCHITECTURE STYLES ==========
Layered (N-Tier) → 6 matched NFRs
Event-Driven/ Messaging → 5 matched NFRs
Component Based → 4 matched NFRs
Serverless/FaaS → 4 matched NFRs
Event-Bus/Event Broker → 4 matched NFRs

Saved results to Ordinal_Method_Top_Arch.json


In [31]:
texts = [
    "Adding more servers to handle 10x users on an e-commerce site",
    "Adding a new payment method in a banking app without breaking existing features" 
]

tokens = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
with torch.no_grad():
    outputs = model(**tokens)
    preds = torch.argmax(outputs.logits, dim=1)

pred_labels = le.inverse_transform(preds.numpy())
print("Predicted NFR categories:", pred_labels.tolist())

Predicted NFR categories: ['SC', 'MN']


#Hena el binary 

In [32]:
%pip install transformers torch pandas numpy scikit-learn

import pandas as pd
import numpy as np
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
import torch
import json
import os


Note: you may need to restart the kernel to use updated packages.


In [33]:
df = pd.read_csv("nfr_training_Binarydata.csv")

print("Dataset loaded:")
df.head()


Dataset loaded:


,category,text,label
0,Performance,The system must provide a fast and efficient p...,1
1,Performance,CV parsing speed must be optimized to under 2 ...,1
2,Performance,System should respond in less than 1 second.,1
3,Performance,System performance can vary under load.,1
4,Performance,Performance is not critical for this system.,0


In [34]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def encode(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

encoded = df.copy()
encoded = encoded[:5000]  # optional safety slice

from datasets import Dataset
dataset = Dataset.from_pandas(encoded)
dataset = dataset.map(encode, batched=True)

dataset = dataset.rename_column("label", "labels")
dataset.set_format(type="torch", columns=["input_ids","attention_mask","labels"])


Map: 100%|██████████| 792/792 [00:00<00:00, 3531.05 examples/s]


In [35]:
train_dataset, test_dataset = dataset.train_test_split(test_size=0.2).values()
len(train_dataset), len(test_dataset)



(633, 159)

In [36]:
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [37]:
import inspect

args_kwargs = {
    "output_dir": "./results_binary",
    "learning_rate": 2e-5,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 8,
    "per_device_eval_batch_size": 8,
    "logging_dir": "./logs",
    "logging_steps": 20,
}

sig = inspect.signature(TrainingArguments.__init__)
if "evaluation_strategy" in sig.parameters:
    args_kwargs["evaluation_strategy"] = "epoch"
elif "eval_steps" in sig.parameters:
    # approximate one evaluation per epoch
    steps_per_epoch = max(1, len(train_dataset) // args_kwargs["per_device_train_batch_size"])
    args_kwargs["eval_steps"] = steps_per_epoch
if "do_eval" in sig.parameters:
    args_kwargs["do_eval"] = True

training_args = TrainingArguments(**args_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()


c:\Users\moham\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
20,0.656100
40,0.516300
60,0.395500
80,0.410800
100,0.230800
120,0.188900
140,0.274300
160,0.122400
180,0.100800
200,0.148400


TrainOutput(global_step=240, training_loss=0.2671379193663597, metrics={'train_runtime': 1087.5782, 'train_samples_per_second': 1.746, 'train_steps_per_second': 0.221, 'total_flos': 124911973532160.0, 'train_loss': 0.2671379193663597, 'epoch': 3.0})

In [38]:
from transformers import BertTokenizer, BertForSequenceClassification

# 👇 Use the same tokenizer & model objects you used for training
SAVE_DIR = "trained_nfr_binary_model"   # pick any name you like

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"Model and tokenizer saved to folder: {SAVE_DIR}")


Model and tokenizer saved to folder: trained_nfr_binary_model


In [39]:
def predict_nfr(text):
    tokens = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=128)
    with torch.no_grad():
        output = model(**tokens)
    pred = torch.argmax(output.logits).item()
    return pred  # 0 or 1


In [40]:
arch_df = pd.read_csv("architecture_datasetBinary (1).csv")

print("Architecture dataset loaded:")
arch_df.head()


Architecture dataset loaded:


,Architecture Style,PE,SC,MN,A,SE,US,PO,O,Unnamed: 9,Unnamed: 10
0,Monolithic,1,0,0,1,1,1,0,1,NaN,NaN
1,Client server,1,1,1,1,1,1,1,1,NaN,NaN
2,Microservices,1,1,1,1,1,1,1,0,NaN,NaN
3,Service Oriented (SOA),1,1,1,1,1,1,1,1,NaN,NaN
4,Event-Driven/ Messaging,1,1,1,1,1,1,1,1,NaN,NaN


In [41]:
NFR_ORDER = ["PE","SC","MN","A","SE","US","PO","O"]


In [42]:
def srs_to_binary_vector(srs_sentences):
    vector = {k: 0 for k in NFR_ORDER}  

    for sentence in srs_sentences:
        for nfr_cat in NFR_ORDER:
            if nfr_cat.lower() in sentence.lower():  
                vector[nfr_cat] = predict_nfr(sentence)

    return vector


In [43]:
def compute_arch_scores(srs_vector):
    results = []

    for _, row in arch_df.iterrows():
        arch = row["Architecture Style"]

        arch_vec = row[NFR_ORDER].values.astype(int)
        srs_vec = np.array([srs_vector[k] for k in NFR_ORDER], dtype=int)

        diff = np.sum(np.abs(arch_vec - srs_vec))
        score = 1 - (diff / len(NFR_ORDER))  # normalized score 0–1

        results.append((arch, score))

    results.sort(key=lambda x: x[1], reverse=True)

    return results


In [44]:
import json

# Load extracted NFRs from main.py output
with open("non_functional_requirements.json", "r", encoding="utf-8") as f:
    extracted_nfrs = json.load(f)

# Extract the "description" field from each NFR object
srs = [item["description"] for item in extracted_nfrs]

print("NFRs loaded from JSON:")
print(srs)


NFRs loaded from JSON:
['The system shall comply with relevant standards and regulations, including accessibility standards and data privacy laws.', 'The system shall be accessible to individuals with Dyslexia, including those with visual, auditory, motor, or cognitive disabilities.', 'The system shall be compatible with a variety of devices, including desktop computers, laptops, tablets, and smartphones.', "The system shall comply with relevant data privacy laws and regulations, including the General Data Protection Regulation (GDPR) and the Children's Online Privacy Protection Act (COPPA).", 'The system shall be scalable to accommodate a large number of users and provide a seamless user experience.', 'The system shall be user-friendly and easy to navigate, with clear and concise instructions and feedback.', 'The system shall provide fast and responsive performance, with minimal latency and downtime.', 'The system shall provide robust security measures to protect user data and prevent

In [45]:
# Load extracted NFR sentences from the JSON file created by main.py
with open("non_functional_requirements.json", "r", encoding="utf-8") as f:
    extracted_nfr = json.load(f)

# Convert extracted NFR list into binary vector
srs_vector = srs_to_binary_vector([item["description"] for item in extracted_nfr])

# Compute architecture ranking
scores = compute_arch_scores(srs_vector)

print("Top 5 architectures:")
for arch, score in scores[:5]:
    print(arch, score)

print("\nBest Architecture:", scores[0][0])


Top 5 architectures:
Client server 0.875
Service Oriented (SOA) 0.875
Event-Driven/ Messaging 0.875
Component Based 0.875
REST/ resource-oriented 0.875

Best Architecture: Client server


In [46]:
import os
import json


# Create final save path
save_path = os.path.join("Binary_method_Top_arch.json")

# Build output dictionary
output = {
    "srs_vector": srs_vector,
    "top_5_architectures": scores[:5],
    "best_architecture": scores[0]
}

# Save output file INSIDE your project folder
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=4, ensure_ascii=False)

print(f"Saved to: {save_path}")


Saved to: Binary_method_Top_arch.json


In [69]:
# ============================================================
# Cell 1: Imports + Seed Setup
# ============================================================

import torch
import pandas as pd
import json
import os
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.preprocessing import LabelEncoder
import numpy as np
import random

# تثبيت الـ seed
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [70]:
# ============================================================
# Cell 2: Load fine-tuned BERT model
# ============================================================

MODEL_DIR = "./trained_nfr_model"

tokenizer = BertTokenizer.from_pretrained(MODEL_DIR)
model = BertForSequenceClassification.from_pretrained(MODEL_DIR)
model.eval()


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [71]:
# ============================================================
# Cell 3: Load training CSV for LabelEncoder
# ============================================================

TRAIN_CSV = "merged_NFR_cleaned.csv"

df = pd.read_csv(TRAIN_CSV)

le = LabelEncoder()
le.fit(df["Type"])

NFR_CATEGORIES = list(le.classes_)   # Example: ["Performance", "Security", ...]
NFR_CATEGORIES


['A', 'FT', 'L', 'LF', 'MN', 'O', 'PE', 'PO', 'SC', 'SE', 'US']

In [72]:
# ============================================================
# Cell 4: Load Extracted NFR JSON
# ============================================================

JSON_PATH = "non_functional_requirements.json"

if not os.path.exists(JSON_PATH):
    raise FileNotFoundError(f"JSON not found: {JSON_PATH}")

with open(JSON_PATH, "r", encoding="utf-8") as f:
    EXTRACTED_NFRS = json.load(f)

EXTRACTED_NFRS[:5]   # preview


[{'title': 'Performance',
  'description': 'MUST respond to user input within 2 seconds.',
  'source': {'page': None, 'start_index': 21}},
 {'title': 'Reliability',
  'description': 'SHALL be available 99.9% of the time.',
  'source': {'page': None, 'start_index': 21}},
 {'title': 'Security',
  'description': 'MUST protect user data from unauthorized access.',
  'source': {'page': None, 'start_index': 21}},
 {'title': 'Privacy',
  'description': 'SHALL ensure that user data is collected and stored in accordance with relevant laws and regulations.',
  'source': {'page': None, 'start_index': 21}},
 {'title': 'Scalability',
  'description': 'SHOULD be able to handle a minimum of 10,000 users concurrently.',
  'source': {'page': None, 'start_index': 21}}]

In [73]:
# ============================================================
# Cell 5: Requirement Strength Function
# ============================================================

def requirement_strength(description):
    s = description.lower()
    if "must" in s:
        return 3.0
    if "shall" in s:
        return 2.0
    if "should" in s:
        return 1.0
    return 0.5  # weak requirement


In [74]:
# ============================================================
# Cell 6: Predict NFR Type using Fine-Tuned BERT
# ============================================================

def predict_nfr(sentence):
    tokens = tokenizer(
        sentence,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )
    with torch.no_grad():
        outputs = model(**tokens)
        pred_idx = torch.argmax(outputs.logits, dim=1).item()
    return le.inverse_transform([pred_idx])[0]


In [54]:
import sys
!{sys.executable} -m pip install nltk


  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.5 MB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 5.0 MB/s  0:00:00
Using cached click-8.3.1-py3-none-any.whl (108 kB)

   ---------------------------------------- 0/2 [click]
   ---------------------------------------- 0/2 [click]
   ---------------------------------------- 0/2 [click]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/

In [56]:
import sys
!{sys.executable} -m pip install PyPDF2


  Using cached pypdf2-3.0.1-py3-none-any.whl.metadata (6.8 kB)
Using cached pypdf2-3.0.1-py3-none-any.whl (232 kB)


In [63]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")   # important for newer NLTK


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\moham\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\moham\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [75]:
# ============================================================
# Compute Importance Score from SRS.pdf inside /uploads folder
# ============================================================

import nltk
import torch
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.preprocessing import LabelEncoder
import PyPDF2
import os

# ------------------------------------------------------------
# 1. Load BERT model and tokenizer
# ------------------------------------------------------------
MODEL_DIR = "./trained_nfr_model"
tokenizer = BertTokenizer.from_pretrained(MODEL_DIR)
model = BertForSequenceClassification.from_pretrained(MODEL_DIR)
model.eval()

# ------------------------------------------------------------
# 2. Reload label encoder from training CSV
# ------------------------------------------------------------
TRAIN_CSV = "merged_NFR_cleaned.csv"
df = pd.read_csv(TRAIN_CSV)

le = LabelEncoder()
le.fit(df["Type"])
NFR_TYPES = list(le.classes_)

# ------------------------------------------------------------
# 3. Read SRS PDF from /uploads folder
# ------------------------------------------------------------
def load_pdf_text(pdf_path):
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        text = ""
        for page in reader.pages:
            text += page.extract_text() + "\n"
    return text


# ------------------------------------------------------------
# 4. Split text into sentences
# ------------------------------------------------------------
def split_into_sentences(text):
    nltk.download("punkt", quiet=True)
    return nltk.sent_tokenize(text)


# ------------------------------------------------------------
# 5. Predict NFR for each sentence
# ------------------------------------------------------------
def predict_nfr(sentence):
    tokens = tokenizer(
        sentence,
        truncation=True,
        padding=True,
        max_length=256,
        return_tensors="pt"
    )
    with torch.no_grad():
        outputs = model(**tokens)
        pred_index = torch.argmax(outputs.logits, dim=1).item()
    return le.inverse_transform([pred_index])[0]


# ------------------------------------------------------------
# 6. Compute importance score from entire SRS
# ------------------------------------------------------------
def compute_importance_from_srs_text(full_srs_text):

    sentences = split_into_sentences(full_srs_text)

    counts = {t: 0 for t in NFR_TYPES}
    total_sentences = len(sentences)

    for s in sentences:
        nfr = predict_nfr(s)
        counts[nfr] += 1

    importance = {
        k: round(v / total_sentences, 4) for k, v in counts.items()
    }

    return importance, counts, total_sentences


# ------------------------------------------------------------
# 7. Run the pipeline (change filename if needed)
# ------------------------------------------------------------
PDF_PATH = "uploads/SRS.pdf"  # <<=== HERE

srs_text = load_pdf_text(PDF_PATH)
importance, counts, total = compute_importance_from_srs_text(srs_text)

print("\n=== IMPORTANCE SCORES FROM FULL SRS ===")
print(importance)
print("\n=== RAW COUNTS ===")
print(counts)
print("\nTotal sentences:", total)



=== IMPORTANCE SCORES FROM FULL SRS ===
{'A': 0.0066, 'FT': 0.0, 'L': 0.0008, 'LF': 0.0066, 'MN': 0.0272, 'O': 0.0248, 'PE': 0.0066, 'PO': 0.0099, 'SC': 0.0091, 'SE': 0.0231, 'US': 0.8853}

=== RAW COUNTS ===
{'A': 8, 'FT': 0, 'L': 1, 'LF': 8, 'MN': 33, 'O': 30, 'PE': 8, 'PO': 12, 'SC': 11, 'SE': 28, 'US': 1073}

Total sentences: 1212


In [76]:
# ============================================================
# Cell 7: Compute NFR Frequency, Must Score, Importance
# ============================================================

def compute_nfr_scores_from_extraction(extracted_list):
    freq_counts = {cat: 0 for cat in NFR_CATEGORIES}
    must_scores = {cat: 0.0 for cat in NFR_CATEGORIES}

    predicted_nfrs = []

    for item in extracted_list:
        desc = item["description"]
        nfr_type = predict_nfr(desc)

        predicted_nfrs.append(nfr_type)
        strength = requirement_strength(desc)

        freq_counts[nfr_type] += 1
        must_scores[nfr_type] += strength

    # Keep only categories that appear
    existing_nfrs = set(predicted_nfrs)
    freq_counts = {k: v for k, v in freq_counts.items() if k in existing_nfrs}
    must_scores = {k: v for k, v in must_scores.items() if k in existing_nfrs}

    max_freq = max(freq_counts.values()) or 1
    freq_norm = {k: v / max_freq for k, v in freq_counts.items()}

    max_must = max(must_scores.values()) or 1
    must_norm = {k: v / max_must for k, v in must_scores.items()}

    total = sum(freq_counts.values()) or 1


    return freq_norm, must_norm, importance


In [77]:
# ============================================================
# Cell 8: Compute Total NFR Weight
# ============================================================

def compute_total_nfr_weight(freq_norm, must_norm, importance, weights=(0.333,0.333,0.333)):
    total_weight = {}

    for nfr in freq_norm.keys():
        total_weight[nfr] = (
            weights[0] * freq_norm.get(nfr, 0) +
            weights[1] * must_norm.get(nfr, 0) +
            weights[2] * importance.get(nfr, 0)
        )

    s = sum(total_weight.values()) or 1
    total_weight = {k: round(v / s, 4) for k, v in total_weight.items()}

    return total_weight


In [78]:
# ============================================================
# Cell 9: Compute Top Architectures
# ============================================================

def compute_top_architectures(total_nfr_weights, arch_csv="ArchitectureDataset2.csv", top_n=5):
    df_arch = pd.read_csv(arch_csv)

    arch_scores = {}

    for _, row in df_arch.iterrows():
        arch = row["Architecture"]
        nfr = row["Type"]

        if nfr not in total_nfr_weights:
            continue

        level = row.get("LevelNorm", row["LevelNorm"])
        weight = total_nfr_weights[nfr]

        arch_scores[arch] = arch_scores.get(arch, 0) + level * weight

    total_val = sum(arch_scores.values()) or 1
    arch_scores = {k: round(v / total_val, 4) for k, v in arch_scores.items()}

    return sorted(
        [{"Architecture": k, "Score": v} for k, v in arch_scores.items()],
        key=lambda x: x["Score"],
        reverse=True
    )[:top_n]


In [79]:
# ============================================================
# Cell 10: Full Pipeline
# ============================================================
import json

def compute_all_weights_and_architectures():
    freq_norm, must_norm, importance = compute_nfr_scores_from_extraction(EXTRACTED_NFRS)
    total = compute_total_nfr_weight(freq_norm, must_norm, importance)
    top_arch = compute_top_architectures(total)

    return {
        "normalized_frequency": freq_norm,
        "normalized_must_score": must_norm,
        "importance_score": importance,
        "total_nfr_weights": total,
        "top_architectures": top_arch
    }

results = compute_all_weights_and_architectures()
results
# Save to JSON
with open("Weighted_Score_Top_Arch.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

# Optionally, print to confirm
print("Saved results to Weighted_Score_Top_Arch.json")

Saved results to Weighted_Score_Top_Arch.json
